# 06 · Multi-Head Attention — The Reshape Dance

Companion to **Chapter 6**. The single most important cell in this notebook is
**§2, the loop-vs-vectorised assert**. If that passes, your reshape is right.
If it fails, everything downstream in the course will be subtly broken.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
B, T, C, NH = 2, 12, 96, 6
HD = C // NH
print(f"B={B} T={T} d_model={C} n_head={NH} d_head={HD}")

## 1 · The dance, step by step

Nothing is *computed* here. The same numbers are re-labelled and re-ordered so that
a batched matmul treats each head as an independent problem.

In [ ]:
x = torch.randn(B, T, C)
W_q = nn.Linear(C, C, bias=False)

q = W_q(x)
print(f"1. after W_q          {tuple(q.shape)}")

q = q.view(B, T, NH, HD)
print(f"2. .view(B,T,nh,hd)   {tuple(q.shape)}   <- {C} = {NH} x {HD}, re-labelled only")

q = q.transpose(1, 2)
print(f"3. .transpose(1,2)    {tuple(q.shape)}   <- (B, n_head, T, d_head)")
print(f"   contiguous? {q.is_contiguous()}   <- transpose only changed strides")
print(f"\n   The last two axes are now a ({T}, {HD}) matrix per (batch, head):")
print(f"   {B} x {NH} = {B*NH} independent attention problems.")

## 2 · THE TEST — loop vs vectorised

The definitive proof that your heads are actually independent and correctly sliced.

In [ ]:
W_k, W_v, W_o = (nn.Linear(C, C, bias=False) for _ in range(3))
causal = torch.ones(T, T, dtype=torch.bool).tril()

def mha_loop(x):
    """Explicitly run single-head attention nh times on feature slices."""
    q, k, v = W_q(x), W_k(x), W_v(x)               # 3 x (B, T, C)
    outs = []
    for h in range(NH):
        sl = slice(h * HD, (h + 1) * HD)           # THIS head's feature slice
        qh, kh, vh = q[..., sl], k[..., sl], v[..., sl]      # (B, T, HD)
        s = qh @ kh.transpose(-2, -1) / math.sqrt(HD)
        s = s.masked_fill(~causal, torch.finfo(s.dtype).min)
        outs.append(s.softmax(-1) @ vh)            # (B, T, HD)
    return W_o(torch.cat(outs, dim=-1))            # (B, T, C)

def mha_vectorised(x):
    q, k, v = (f(x).view(B, T, NH, HD).transpose(1, 2) for f in (W_q, W_k, W_v))
    s = q @ k.transpose(-2, -1) / math.sqrt(HD)    # (B, NH, T, T)
    s = s.masked_fill(~causal, torch.finfo(s.dtype).min)
    y = s.softmax(-1) @ v                          # (B, NH, T, HD)
    y = y.transpose(1, 2).reshape(B, T, C)         # transpose FIRST, then reshape
    return W_o(y)

a, b = mha_loop(x), mha_vectorised(x)
print("max abs diff:", (a - b).abs().max().item())
assert torch.allclose(a, b, atol=1e-5), "YOUR RESHAPE IS WRONG -- fix before continuing"
print("loop == vectorised ✓   your heads are correctly independent")

## 3 · The bug: forgetting the transpose

`y.reshape(B,T,C)` without the transpose runs fine and scrambles everything.

In [ ]:
q, k, v = (f(x).view(B, T, NH, HD).transpose(1, 2) for f in (W_q, W_k, W_v))
s = (q @ k.transpose(-2, -1) / math.sqrt(HD)).masked_fill(~causal, torch.finfo(x.dtype).min)
y = s.softmax(-1) @ v                              # (B, NH, T, HD)

right = y.transpose(1, 2).reshape(B, T, C)
wrong = y.reshape(B, T, C)                         # no error!

print(f"both are {tuple(right.shape)} -- no exception either way")
print(f"identical? {torch.allclose(right, wrong)}")
print(f"max diff:  {(right - wrong).abs().max().item():.4f}")
assert not torch.allclose(right, wrong)
print("\nThe wrong one mixes OTHER TOKENS' head outputs into each token's vector.")
print("It trains. To a mediocre loss. And never tells you why.")

## 4 · Exercise 6.2 — the einops rewrite

Makes the bug above *impossible to write by accident*.

In [ ]:
try:
    from einops import rearrange

    def mha_einops(x):
        q, k, v = (rearrange(f(x), 'b t (h d) -> b h t d', h=NH)
                   for f in (W_q, W_k, W_v))
        s = torch.einsum('bhqd,bhkd->bhqk', q, k) / math.sqrt(HD)
        s = s.masked_fill(~causal, torch.finfo(s.dtype).min)
        y = torch.einsum('bhqk,bhkd->bhqd', s.softmax(-1), v)
        return W_o(rearrange(y, 'b h t d -> b t (h d)'))

    assert torch.allclose(mha_einops(x), mha_vectorised(x), atol=1e-5)
    print("einops version matches ✓")
    print("\n  'b h t d -> b t (h d)'  states the intent explicitly.")
    print("  'b h t d -> b t (d h)'  would be visibly different -- not a silent bug.")
except ImportError:
    print("einops not installed:  pip install einops")
    print("(optional, but worth it -- see Chapter 2.4)")

## 5 · Multi-head attention is FREE

Same parameters, same FLOPs, `n_head` independent attention patterns.

In [ ]:
print(f"{'n_head':>8} {'d_head':>8} {'params':>12} {'score elems':>13}")
for nh in [1, 2, 4, 6, 12]:
    if C % nh: continue
    hd = C // nh
    params = 4 * C * C                    # W_q, W_k, W_v, W_o -- independent of nh
    score_elems = nh * T * T              # more heads, smaller each... 
    print(f"{nh:>8} {hd:>8} {params:>12,} {score_elems:>13,}")

print("\nParameters do not depend on n_head at all: d_model is PARTITIONED,")
print("not duplicated. You spend representational width to buy relational diversity.")

## 6 · Exercise 6.4 — parameter accounting with GQA

In [ ]:
def attn_params(d_model, n_head, n_kv_head, d_head):
    wq = d_model * n_head * d_head
    wk = d_model * n_kv_head * d_head
    wv = d_model * n_kv_head * d_head
    wo = n_head * d_head * d_model
    return wq, wk, wv, wo

print(f"{'config':>28} {'W_q':>12} {'W_k':>10} {'W_v':>10} {'W_o':>12} {'total':>12}")
for label, nh, kv, hd in [("MHA  32 heads, d_head 128", 32, 32, 128),
                          ("MHA  64 heads, d_head 64",  64, 64,  64),
                          ("GQA  32 q / 8 kv",          32,  8, 128),
                          ("MQA  32 q / 1 kv",          32,  1, 128)]:
    p = attn_params(4096, nh, kv, hd)
    print(f"{label:>28} {p[0]:>12,} {p[1]:>10,} {p[2]:>10,} {p[3]:>12,} {sum(p):>12,}")

print("\nRows 1 and 2 are IDENTICAL -- the head split is free.")
print("GQA cuts attention params ~37% -- but the real prize is a 4x smaller")
print("KV CACHE, which is what actually limits serving. That is Chapter 16.")

---
## Self-check

1. `d_model=768, n_head=12, B=4, T=256`. Shape after `view(...).transpose(1,2)`?
2. Does 12-head attention have more parameters than 1-head at the same `d_model`?
3. What breaks if you `reshape` before `transpose` when merging heads?
4. Why does `W_o` exist?

<details><summary>Answers</summary>

1. `(4, 12, 256, 64)` — `(B, n_head, T, d_head)`.
2. No. Identical. `d_model` is partitioned across heads.
3. Each token's output vector gets built from *other tokens'* head outputs. Valid
   shapes, no error, quietly wrong model.
4. After concatenation the heads occupy disjoint feature blocks that have never
   interacted. `W_o` lets head 3's output combine with head 7's before being
   written back to the residual stream.

</details>

**Next:** `09_full_model.ipynb`